In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr_sk
from skimage.metrics import structural_similarity as ssim_sk

In [ ]:
IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_CHANNELS = 3
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 196

MASK_RATIO = 0.75
NUM_VISIBLE = int(NUM_PATCHES * (1 - MASK_RATIO))  # 49

# encoder parameters
ENC_EMBED_DIM = 768
ENC_DEPTH = 12
ENC_NUM_HEADS = 12

# decoder parameters
DEC_EMBED_DIM = 384
DEC_DEPTH = 12
DEC_NUM_HEADS = 6

In [ ]:
class TinyImageNet(Dataset):
    """
    Load images from Tiny ImageNet-200.
    Expects folder structure:
      tiny-imagenet-200/train/n01443507/images/*.JPEG
      tiny-imagenet-200/val/images/*.JPEG
    """

    def __init__(self, root, split="train"):
        """
        root: path to tiny-imagenet-200 folder
        split: "train" or "val"
        """
        self.root = Path(root)
        self.split = split
        self.samples = []

        if split == "train":
            # Each class has an images/ subfolder
            for class_dir in (self.root / "train").iterdir():
                if not class_dir.is_dir():
                    continue
                img_dir = class_dir / "images"
                if img_dir.exists():
                    for p in img_dir.glob("*.JPEG"):
                        self.samples.append(str(p))
        else:
            # Val: images are in val/images/
            img_dir = self.root / "val" / "images"
            if img_dir.exists():
                self.samples = [str(p) for p in img_dir.glob("*.JPEG")]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
        img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        return img


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.patch_size = PATCH_SIZE
        self.num_patches = NUM_PATCHES

        # Each patch: 16*16*3 = 768 values -> one vector of size embed_dim (which os 768)
        patch_dim = PATCH_SIZE * PATCH_SIZE * NUM_CHANNELS
        self.proj = nn.Linear(patch_dim, embed_dim)

    def forward(self, x):
        # x: (B, 3, 224, 224)
        B, C, H, W = x.shape
        p = self.patch_size
        assert H == W == IMAGE_SIZE

        # reshape into patches: (B, 3, 14, 16, 14, 16) -> (B, 14*14, 16*16*3)
        x = x.reshape(B, C, H // p, p, W // p, p)
        x = x.permute(0, 2, 4, 3, 5, 1)  # (B, 14, 14, 16, 16, 3)
        x = x.reshape(B, self.num_patches, -1)  # (B, 196, 768)

        return self.proj(x)  # (B, 196, embed_dim)

# vit base
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbedding(ENC_EMBED_DIM)
        self.num_patches = NUM_PATCHES

        # learnable positional embeddings (one per patch position)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, ENC_EMBED_DIM))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # stack of transformer layers (built-in by pytorch, pls dont touch these muzammil)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                ENC_EMBED_DIM,
                ENC_NUM_HEADS,
                batch_first=True,
                norm_first=True,
            )
            for _ in range(ENC_DEPTH)
        ])
        self.norm = nn.LayerNorm(ENC_EMBED_DIM)

    def forward(self, x, visible_indices):
        """
        x: (B, 3, 224, 224) — full image
        visible_indices: (B, num_visible) — which patch indices are visible (no mask tokens)
        Returns: (B, num_visible, ENC_EMBED_DIM) — latent only for visible patches
        """
        # patch embed full image so we can select visible patches
        tokens = self.patch_embed(x)  # (B, 196, 768)

        # add positional embedding (same for all samples; we index by position)
        tokens = tokens + self.pos_embed  # (B, 196, 768)

        # Keep only visible tokens, encoder never sees masked patches
        # visible_indices: (B, 49) -> gather tokens at those positions
        B, N, D = tokens.shape
        visible_indices = visible_indices.unsqueeze(-1).expand(-1, -1, D)  # (B, 49, 768)
        tokens = torch.gather(tokens, 1, visible_indices)  # (B, 49, 768)

        # Transformer layers
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)
        return tokens

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_patches = NUM_PATCHES

        # encoder outputs are 768-dim; we project to decoder dim 384
        self.enc_to_dec = nn.Linear(ENC_EMBED_DIM, DEC_EMBED_DIM)

        # learnable mask tokens (one vector per masked position, same for all positions)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, DEC_EMBED_DIM))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        # positional embeddings in decoder (so it knows patch order for reconstruction)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, DEC_EMBED_DIM))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                DEC_EMBED_DIM,
                DEC_NUM_HEADS,
                batch_first=True,
                norm_first=True,
            )
            for _ in range(DEC_DEPTH)
        ])
        self.norm = nn.LayerNorm(DEC_EMBED_DIM)

        # predict patch pixels: each decoder output = 16*16*3 values
        self.head = nn.Linear(DEC_EMBED_DIM, PATCH_SIZE * PATCH_SIZE * NUM_CHANNELS)

    def forward(self, latent, visible_indices, mask_indices):
        """
        latent: (B, num_visible, ENC_EMBED_DIM) from encoder
        visible_indices: (B, num_visible): positions of visible patches
        mask_indices: (B, num_masked): positions we must reconstruct
        Returns: (B, num_masked, patch_pixels): predicted pixel values for masked patches only
        """
        B, num_visible, _ = latent.shape
        num_masked = mask_indices.shape[1]

        # project encoder latent to decoder dimension
        latent = self.enc_to_dec(latent)  # (B, 49, 384)

        # build full sequence in original patch order: put latent at visible positions,
        # mask_token at masked positions
        tokens = self.mask_token.expand(B, self.num_patches, -1).clone()  # (B, 196, 384)
        visible_indices_exp = visible_indices.unsqueeze(-1).expand(-1, -1, DEC_EMBED_DIM)
        tokens.scatter_(1, visible_indices_exp, latent)

        # add positional embeddings
        tokens = tokens + self.pos_embed

        # transformer layers
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)

        # predict pixels for all patches, then keep only masked positions for loss
        pred_all = self.head(tokens)  # (B, 196, patch_pixels)
        mask_indices_exp = mask_indices.unsqueeze(-1).expand(-1, -1, pred_all.shape[-1])
        pred_masked = torch.gather(pred_all, 1, mask_indices_exp)  # (B, num_masked, patch_pixels)
        return pred_masked

class MAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.num_patches = NUM_PATCHES
        self.num_visible = NUM_VISIBLE
        self.mask_ratio = 1 - (NUM_VISIBLE / NUM_PATCHES)

    def _random_mask(self, B, device):
        """
        For each sample, shuffle patch indices and split into visible / masked.
        Returns:
            visible_indices: (B, num_visible)
            mask_indices: (B, num_masked)
        """
        num_masked = self.num_patches - self.num_visible
        # Random permutation per sample
        rand = torch.rand(B, self.num_patches, device=device)
        ids = rand.argsort(dim=1)  # (B, 196)
        visible_indices = ids[:, :self.num_visible]      # (B, 49): first 49 will be visible
        mask_indices = ids[:, self.num_visible:]         # (B, 147): last 147 will be hiddel
        return visible_indices, mask_indices

    def forward(self, x, visible_indices=None, mask_indices=None):
        """
        x: (B, 3, 224, 224)
        """
        B = x.shape[0]
        device = x.device
        if visible_indices is None or mask_indices is None:
            visible_indices, mask_indices = self._random_mask(B, device)

        # only visible patches -> latent rep
        latent = self.encoder(x, visible_indices)  # (B, 49, 768)

        # latent rep + mask tokens -> predict masked patch pixels
        pred_masked = self.decoder(latent, visible_indices, mask_indices)  # (B, 147, 768)
        return pred_masked, visible_indices, mask_indices

# helper for loss
def image_to_patch_pixels(x):
    """
    turns image into patch pixels (no projection). Same grid as PatchEmbedding.
    x: (B, 3, 224, 224) -> out: (B, 196, 16*16*3)
    """
    B, C, H, W = x.shape
    p = PATCH_SIZE
    x = x.reshape(B, C, H // p, p, W // p, p)
    x = x.permute(0, 2, 4, 3, 5, 1)
    return x.reshape(B, NUM_PATCHES, -1)


def patch_pixels_to_image(patches):
    """
    inverse of image_to_patch_pixels. Reassemble patch pixels into image.
    patches: (B, 196, 16*16*3) -> out: (B, 3, 224, 224)
    """
    B, N, D = patches.shape
    p = PATCH_SIZE
    side = (N ** 0.5)
    assert side == int(side), "NUM_PATCHES must be a perfect square"
    side = int(side)
    # (B, 196, 768) -> (B, 14, 14, 16, 16, 3)
    patches = patches.reshape(B, side, side, p, p, NUM_CHANNELS)
    # (B, 14, 14, 16, 16, 3) -> (B, 3, 14, 16, 14, 16)
    x = patches.permute(0, 5, 1, 3, 2, 4)
    return x.reshape(B, NUM_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

def mae_loss(model_out, target_patches, mask_indices):
    """
    model_out: (B, num_masked, patch_pixels): model prediction for masked patches (normalized space)
    target_patches: (B, 196, patch_pixels): ground-truth patch pixels from image
    mask_indices: (B, num_masked): which positions were masked

    patch normalization (per-patch mean/var) so the model learns structure rather than average brightness, improving reconstruction quality (mae paper says so).
    """
    B, num_masked, patch_dim = model_out.shape
    mask_exp = mask_indices.unsqueeze(-1).expand(-1, -1, patch_dim)
    target_masked = torch.gather(target_patches, 1, mask_exp)  # (B, num_masked, patch_dim)

    # per-patch normalization: equalize brightness/contrast so MSE focuses on structure
    mean = target_masked.mean(dim=-1, keepdim=True)
    var = target_masked.var(dim=-1, keepdim=True)
    target_masked = (target_masked - mean) / torch.sqrt(var + 1e-6)

    return nn.functional.mse_loss(model_out, target_masked)

In [ ]:
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    fraction = 0.1  # training on 1% of train set
    full_train = TinyImageNet("tiny-imagenet-200", split="train")
    full_val = TinyImageNet("tiny-imagenet-200", split="val")
    n_train = len(full_train)
    n_val = len(full_val)
    train_set = Subset(full_train, torch.randperm(n_train)[:max(1, int(n_train * fraction))].tolist())
    val_set = Subset(full_val, torch.randperm(n_val)[:max(1, int(n_val * fraction))].tolist())
    train_loader = DataLoader(train_set, batch_size=8)
    val_loader = DataLoader(val_set, batch_size=8)
    print(f"Train: {len(train_set)} ({100*fraction:.0f}% of {n_train})  Val: {len(val_set)} ({100*fraction:.0f}% of {n_val})")

    model = MAE().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    state = torch.load("model.pt", map_location=device, weights_only=True)
    model.load_state_dict(state)
    model = model.to(device)

    model.train()
    for epoch in range(1, 20):
        total_loss = 0.0
        num_batches = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
        for batch in pbar:
            x = batch.to(device)

            pred_masked, visible_indices, mask_indices = model(x)
            target_patches = image_to_patch_pixels(x)
            loss = mae_loss(pred_masked, target_patches, mask_indices)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = total_loss / num_batches

        # Validation
        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Val", leave=False):
                x = batch.to(device)
                pred_masked, _, mask_indices = model(x)
                target_patches = image_to_patch_pixels(x)
                val_loss += mae_loss(pred_masked, target_patches, mask_indices).item()
                val_batches += 1
        model.train()
        val_loss = val_loss / val_batches

        print(f"Epoch {epoch}  train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")
        torch.save(model.state_dict(), "model.pt")
        
    print("Training finished.")

In [ ]:
train()

In [ ]:
def build_masked_image(x, mask_indices, fill=0.5):
    """
    replaces masked patches with a constant (e.g. gray).
    x: (B, 3, H, W), mask_indices: (B, num_masked)
    Returns: (B, 3, H, W) with masked patches set to fill.
    """
    B, C, H, W = x.shape
    target_patches = image_to_patch_pixels(x)  # (B, 196, 768)
    # replace masked positions with gray patch
    patch_dim = target_patches.shape[-1]
    gray_patch = torch.full((1, 1, patch_dim), fill, device=x.device, dtype=x.dtype)
    gray_patch = gray_patch.expand(B, mask_indices.shape[1], -1)
    mask_exp = mask_indices.unsqueeze(-1).expand(-1, -1, patch_dim)
    target_patches = target_patches.clone()
    target_patches.scatter_(1, mask_exp, gray_patch)
    return patch_pixels_to_image(target_patches)


def compute_psnr_ssim(recon, gt, data_range=1.0):
    """
    Compute PSNR (dB) and SSIM between reconstruction and ground truth.
    recon, gt: (C, H, W) tensors in [0, 1].
    Returns: (psnr_db, ssim_value).
    """
    r = np.clip(recon.numpy(), 0.0, 1.0)
    g = np.clip(gt.numpy(), 0.0, 1.0)
    # (C, H, W) -> (H, W, C)
    r = r.transpose(1, 2, 0)
    g = g.transpose(1, 2, 0)
    psnr = psnr_sk(g, r, data_range=data_range)
    ssim = ssim_sk(g, r, data_range=data_range, channel_axis=2)
    return float(psnr), float(ssim)


def build_reconstruction_image(x, pred_masked, visible_indices, mask_indices):
    """
    Full image: visible patches from original, masked patches from model prediction.
    Model predicts normalized patch pixels; we denormalize using target mean/var before
    assembling the image (same normalization as in mae_loss).
    x: (B, 3, H, W), pred_masked: (B, num_masked, patch_pixels) in normalized space
    Returns: (B, 3, H, W).
    """
    target_patches = image_to_patch_pixels(x)  # (B, 196, patch_dim)
    patch_dim = target_patches.shape[-1]
    mask_exp = mask_indices.unsqueeze(-1).expand(-1, -1, patch_dim)
    target_masked = torch.gather(target_patches, 1, mask_exp)

    # Denormalize: model predicts (target - mean) / std
    mean = target_masked.mean(dim=-1, keepdim=True)
    var = target_masked.var(dim=-1, keepdim=True)
    pred_pixels = pred_masked * torch.sqrt(var + 1e-6) + mean

    target_patches = target_patches.clone()
    target_patches.scatter_(1, mask_exp, pred_pixels)
    return patch_pixels_to_image(target_patches)


def visualize_reconstructions(
    model,
    dataloader,
    device,
    num_examples=5,
    save_path="mae_reconstructions.png",
):
    """
    run model on batches, collect 4-panel visuals for at least num_examples samples.
    uses a fixed mask (same seed) for reproducibility.
    """
    model.eval()
    B = 1
    # Fixed mask for all samples (we'll use the same mask pattern per batch position)
    rand = torch.rand(B, NUM_PATCHES, device=device)
    ids = rand.argsort(dim=1)
    num_visible = model.num_visible
    visible_indices = ids[:, :num_visible]
    mask_indices = ids[:, num_visible:]

    collected = []  # list of (masked_im, recon_im, gt_im, psnr, ssim) per sample

    with torch.no_grad():
        for batch in dataloader:
            x = batch.to(device)
            # Expand fixed mask to batch size (same mask for all in batch for simplicity)
            vis = visible_indices.expand(x.shape[0], -1)
            msk = mask_indices.expand(x.shape[0], -1)

            pred_masked, v, m = model(x, visible_indices=vis, mask_indices=msk)
            masked_im = build_masked_image(x, m, fill=0.5)
            recon_im = build_reconstruction_image(x, pred_masked, v, m)

            for i in range(x.shape[0]):
                recon_cpu = recon_im[i].cpu()
                gt_cpu = x[i].cpu()
                p, s = compute_psnr_ssim(recon_cpu, gt_cpu)
                collected.append((
                    masked_im[i].cpu(),
                    recon_cpu,
                    gt_cpu,
                    p,
                    s,
                ))
                if len(collected) >= num_examples:
                    break
            if len(collected) >= num_examples:
                break

    n = len(collected)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    titles = ["Masked Input (75% removed)", "Model Reconstruction", "Original Ground Truth"]
    for i in range(3):
        axes[0, i].set_title(titles[i], fontsize=11)
    for row, (masked, recon, gt, psnr, ssim) in enumerate(collected):
        for col, img in enumerate([masked, recon, gt]):
            ax = axes[row, col]
            # (C, H, W) -> (H, W, C), clip to [0, 1]
            arr = img.permute(1, 2, 0).numpy()
            arr = np.clip(arr, 0.0, 1.0)
            ax.imshow(arr)
            if col == 1:
                ax.text(0.02, 0.98, f"PSNR: {psnr:.2f} dB\nSSIM: {ssim:.4f}",
                        transform=ax.transAxes, fontsize=9, verticalalignment="top",
                        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
            ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    mean_psnr = np.mean([c[3] for c in collected])
    mean_ssim = np.mean([c[4] for c in collected])
    print(f"Saved {n} qualitative examples to {save_path}")
    print(f"Mean PSNR: {mean_psnr:.2f} dB  Mean SSIM: {mean_ssim:.4f}")


def visualize():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Data: small subset for visualization
    full_val = TinyImageNet("tiny-imagenet-200", split="val")
    indices = torch.randperm(len(full_val))[: max(5, 16)].tolist()
    val_set = Subset(full_val, indices)
    val_loader = DataLoader(val_set, batch_size=4)

    model = MAE().to(device)
    ckpt = "model.pt"
    try:
        state = torch.load(ckpt, map_location=device, weights_only=True)
        model.load_state_dict(state)
        print(f"Loaded checkpoint: {ckpt}")
    except FileNotFoundError:
        print(f"No checkpoint found at {ckpt}; using randomly initialized model for demo.")

    visualize_reconstructions(
        model,
        val_loader,
        device,
        num_examples=5,
        save_path="mae_reconstructions.png",
    )


In [ ]:
visualize()